In [ ]:
import kagglehub
# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")
print("Path to dataset files:", path)

In [ ]:
%pip install catboost xgboost tqdm -q

In [ ]:
# Importing libraries
import pandas as pd
import numpy as np
# The Cat
from catboost import CatBoostClassifier, Pool
# sklearn stuff
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix, precision_score, recall_score
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
# drawing and stuff
import matplotlib.pyplot as plt
# basic
import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Task 1: Write your code here:
path = os.path.join(path, 'Q3_data.csv')
data = pd.read_csv(path)

In [ ]:
# Task 2: Write your code here:
data.head()

In [ ]:
# Task 3: Write your code here:
data.info()

In [ ]:
# Task 4: Write your code here:
data.describe()

In [ ]:
# Task 1: Write your code here:

# I took numbers and categorical cols over here
numerical_cols = data.select_dtypes(include=[np.number]).columns
categorical_cols = data.select_dtypes(include=['object', 'category']).columns

# replace empty numerical stuff with median in nums
for col in numerical_cols:
    if data[col].isnull().sum() > 0:
        data[col] = data[col].fillna(data[col].median())

# categorical with mode
for col in categorical_cols:
    if data[col].isnull().sum() > 0:
        data[col] = data[col].fillna(data[col].mode()[0])

In [ ]:
# Task 2: Write your code here:
initial_rows = len(data)
data = data.drop_duplicates()
removed = initial_rows - len(data)
print(f"Dupes: {removed}")

In [ ]:
# Task 3: Write your code here:
if len(categorical_cols) > 0:
    data = pd.get_dummies(data, columns=categorical_cols.tolist(), drop_first=True)
else:
    print("empty... O-O")

In [ ]:
# Task 4: Write your code here:
if 'Target' in data.columns:
    target = data['Target']
    features = data.drop('Target', axis=1)
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)
    data_scaled = pd.DataFrame(features_scaled, columns=features.columns)
    data_scaled['Target'] = target.values
else:
    scaler = StandardScaler()
    data_scaled = pd.DataFrame(scaler.fit_transform(data), columns=data.columns)

In [ ]:
# Task 5: Write your code here:
if 'Target' in data_scaled.columns:
    target_counts = data_scaled['Target'].value_counts()
    ratio = target_counts[0] / target_counts[1] if len(target_counts) > 1 else 0
    print(f"Class distribution: {dict(target_counts)}")
    print(f"Ratio Class 0/Class 1: {ratio:.2f}")
    if ratio > 4 or ratio < 0.25:
        print("Crazyyy imbalance")
    else:
        print("Good to go!")

In [ ]:
# Task 1: Write your code here:
X = data_scaled.drop('Target', axis=1)
y = data_scaled['Target']

In [ ]:
# Task 2,3,4,5: Write your code here:
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

# Storing our data
f1_scores = []
accuracy_scores = []
models = []

# CatBoost parameters
catboost_params = {
    'iterations': 1000,
    'learning_rate': 0.05,
    'depth': 6,
    'l2_leaf_reg': 3,
    'border_count': 128,
    'loss_function': 'Logloss',
    'eval_metric': 'F1',
    'task_type': 'CPU',
    'random_seed': 42,
    'silent': True,
    'auto_class_weights': 'Balanced',
    'early_stopping_rounds': 50
}

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):

    # Split data
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    print(f"Train size: {len(X_train)} (Default: {y_train.sum()}, {y_train.mean()*100:.1f}%)")
    print(f"Validation size: {len(X_val)} (Default: {y_val.sum()}, {y_val.mean()*100:.1f}%)")

    # Creating CatBoost Pool objects
    train_pool = Pool(X_train, y_train)
    val_pool = Pool(X_val, y_val)

    # Train the CatBoost model
    model = CatBoostClassifier(**catboost_params)
    model.fit(train_pool, eval_set=val_pool, use_best_model=True, plot=False)

    # Make predictions
    y_pred = model.predict(X_val)
    y_pred_proba = model.predict_proba(X_val)[:, 1]

    # Calculate metrics
    fold_f1 = f1_score(y_val, y_pred)
    fold_accuracy = accuracy_score(y_val, y_pred)

    f1_scores.append(fold_f1)
    accuracy_scores.append(fold_accuracy)
    models.append(model)

    print(f"F1 Score: {fold_f1:.4f}")
    print(f"Accuracy: {fold_accuracy:.4f}")

    # Additional metrics for better insight
    fold_precision = precision_score(y_val, y_pred)
    fold_recall = recall_score(y_val, y_pred)
    print(f"Precision: {fold_precision:.4f}")
    print(f"Recall: {fold_recall:.4f}")

print(f"Average F1 Score: {np.mean(f1_scores):.4f}")
print(f"Standard deviation: {np.std(f1_scores):.4f}")
print(f"Range: [{min(f1_scores):.4f}, {max(f1_scores):.4f}]")
print(f"Average Accuracy: {np.mean(accuracy_scores):.4f}")
print(f"Standard deviation: {np.std(accuracy_scores):.4f}")

In [ ]:
# first collecting feature importance from all the models
feature_importances = []
for i, model in enumerate(models):
    importances = model.get_feature_importance()
    feature_importances.append(importances)

# Average importance across them folds
avg_importances = np.mean(feature_importances, axis=0)
feature_importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': avg_importances
}).sort_values('Importance', ascending=False)

# This is OUR GOLDEN FEATURE!!
golden_feature = feature_importance_df.iloc[0]
print(f"The Golden Feature is: '{golden_feature}'")

In [ ]:
# I HAVE NO TIME TOT !!